# Haiku Subagent Write Delegation

**Purpose:** When the main agent runs on Sonnet or Opus, every output token is expensive.
Write-heavy tasks (creating multiple files) generate many output tokens containing file
content. This notebook measures whether delegating those writes to a Haiku subagent via
the Task tool saves money — and at what point the savings outweigh the ~40K token
delegation overhead found in `task-subagent-token-overhead.ipynb`.

| # | Question | Proof method |
|---|----------|--------------|
| 1 | Is Haiku write delegation cheaper than Sonnet/Opus doing writes directly? | Compare per-model token costs across N file writes |
| 2 | At what file count does delegation break even? | Scale from 3→5→10 files, find crossover |
| 3 | Is Haiku delegation faster or slower (wall-clock)? | Compare elapsed time per pattern |
| 4 | Does Haiku produce correct file output? | Verify file existence + content after each run |

## Background + Hypothesis

### Verified API pricing (from docs.anthropic.com, Feb 2026)

| Model | Input $/MTok | Output $/MTok | Cache Read $/MTok |
|-------|-------------|--------------|-------------------|
| Haiku 4.5 | $1.00 | $5.00 | $0.10 |
| Sonnet 4.6 | $3.00 | $15.00 | $0.30 |
| Opus 4.6 | $5.00 | $25.00 | $0.50 |

**Output token ratios vs Haiku:** Sonnet = 3×, Opus = 5×.

### Delegation overhead budget

From `task-subagent-token-overhead.ipynb`, Task delegation adds:
- **Parent output**: ~500 tokens (Task tool call JSON + result processing)
- **Subagent system prompt**: ~40K cache tokens × model cache rate
- **Parent extra cache**: ~28K tokens (result processing turn) × parent cache rate

With model-specific cache pricing:
- **Sonnet parent**: overhead ≈ $0.020. Savings = $10/MTok per output token shifted to Haiku.
  Break-even ≈ **2,000 output tokens**.
- **Opus parent**: overhead ≈ $0.027. Savings = $20/MTok per output token shifted.
  Break-even ≈ **1,350 output tokens**.

### Hypothesis

Writing 5 small files (~10 lines each) generates roughly 800–2,000 output tokens.
The break-even is right in this range, which is why we need empirical measurement.

- **H1**: Delegation saves money for ≥5 files on Sonnet, ≥3 files on Opus
- **H2**: Break-even crossover is ~2K output tokens for Sonnet, ~1.35K for Opus
- **H3**: Wall-clock time is similar (Haiku is faster per token, but subprocess adds latency)
- **H4**: Haiku produces correct file output (existence + content)

In [ ]:
import json
import os
import shutil
import statistics
from pathlib import Path
from lib import run_claude

N_RUNS = 3
BASE_DIR = Path("/tmp/claude/haiku-write-test")

# --- Pricing tables ($/MTok) ---
PRICING = {
    "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00, "cache_read": 0.10},
    "claude-sonnet-4-6":        {"input": 3.00, "output": 15.00, "cache_read": 0.30},
    "claude-opus-4-6":          {"input": 5.00, "output": 25.00, "cache_read": 0.50},
}

# Short aliases for lookup fallback
PRICING_ALIASES = {
    "haiku": "claude-haiku-4-5-20251001",
    "sonnet": "claude-sonnet-4-6",
    "opus": "claude-opus-4-6",
}


def _resolve_pricing(model_key):
    """Resolve a model key to its pricing dict."""
    if model_key in PRICING:
        return PRICING[model_key]
    for alias, full_key in PRICING_ALIASES.items():
        if alias in model_key.lower():
            return PRICING[full_key]
    raise KeyError(f"Unknown model: {model_key}")


def extract_usage(cr):
    """Extract token usage from a ClaudeResult."""
    output = cr.output if isinstance(cr.output, dict) else {}
    model_usage = output.get("modelUsage", {})
    num_turns = output.get("num_turns", 0)

    total_input = sum(u.get("inputTokens", 0) for u in model_usage.values())
    total_output = sum(u.get("outputTokens", 0) for u in model_usage.values())
    total_cache = sum(u.get("cacheReadInputTokens", 0) for u in model_usage.values())

    return {
        "inputTokens": total_input,
        "outputTokens": total_output,
        "cacheReadInputTokens": total_cache,
        "total": total_input + total_output + total_cache,
        "num_turns": num_turns,
        "models": list(model_usage.keys()),
        "model_usage": model_usage,
    }


def multi_model_cost_estimate(usage_dict):
    """Compute dollar cost from a modelUsage dict with per-model pricing.

    Args:
        usage_dict: {model_key: {inputTokens, outputTokens, cacheReadInputTokens}}

    Returns:
        {model_key: {input, output, cache_read, total}, "grand_total": float}
    """
    result = {}
    grand_total = 0.0
    for model_key, usage in usage_dict.items():
        prices = _resolve_pricing(model_key)
        inp = usage.get("inputTokens", 0) / 1_000_000 * prices["input"]
        out = usage.get("outputTokens", 0) / 1_000_000 * prices["output"]
        cache = usage.get("cacheReadInputTokens", 0) / 1_000_000 * prices["cache_read"]
        total = inp + out + cache
        result[model_key] = {"input": inp, "output": out, "cache_read": cache, "total": total}
        grand_total += total
    result["grand_total"] = grand_total
    return result


def verify_files(directory, expected_files):
    """Check file existence and non-empty content.

    Args:
        directory: Path to check in.
        expected_files: List of relative filenames.

    Returns:
        {filename: {"exists": bool, "non_empty": bool, "size": int}}
    """
    results = {}
    for fname in expected_files:
        fpath = Path(directory) / fname
        exists = fpath.exists()
        size = fpath.stat().st_size if exists else 0
        results[fname] = {"exists": exists, "non_empty": size > 0, "size": size}
    return results


def run_experiment(prompt, *, model="sonnet", n=N_RUNS, label=""):
    """Run claude -p N times, collect usage stats and timing."""
    results = []
    for i in range(n):
        cr = run_claude(prompt, model=model)
        usage = extract_usage(cr)
        usage["run"] = i + 1
        usage["returncode"] = cr.returncode
        usage["duration_s"] = cr.duration_s
        results.append(usage)
        status = "OK" if cr.returncode == 0 else "FAIL"
        models_str = ", ".join(usage["models"])
        print(f"  [{label}] Run {i+1}/{n}: {status} — "
              f"out={usage['outputTokens']} "
              f"cache={usage['cacheReadInputTokens']} "
              f"time={cr.duration_s:.1f}s "
              f"models=[{models_str}]")
    return results


def summarize(results, metric):
    """Compute mean/stdev/min/max for a metric across runs."""
    values = [r[metric] for r in results]
    return {
        "mean": statistics.mean(values),
        "stdev": statistics.stdev(values) if len(values) > 1 else 0,
        "min": min(values),
        "max": max(values),
        "n": len(values),
    }


def clean_dir(path):
    """Remove and recreate a directory."""
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


# Storage for all results
ALL_RESULTS = {}    # {tier: {"direct": [...], "delegated": [...]}}
ALL_VERIFY = {}     # {tier: {"direct": [...], "delegated": [...]}}

print("Setup ready.")
print(f"Base dir: {BASE_DIR}")
print(f"Runs per experiment: {N_RUNS}")
print(f"Parent model: sonnet (cost-effective proxy)")

## Experiment Design

| Tier | Files | Description | Est. output tokens |
|------|-------|-------------|-------------------|
| Small | 3 | 1-line config files (`key=value`) | ~200 |
| Medium | 5 | Small Python modules (~10 lines each) | ~800 |
| Large | 10 | Python modules + configs + test stubs | ~2000 |

Two patterns per tier (parent model = **sonnet**, as a cost-effective proxy):

- **Pattern A (Direct)**: `claude -p "Write these N files..." --model sonnet`
- **Pattern B (Delegated)**: `claude -p "Use Task tool with model=haiku to write..." --model sonnet`

The parent always runs on Sonnet. In Pattern B, the parent delegates the file-writing
work to a Haiku subagent via the Task tool. We compare:
- Token usage and cost (applying per-model pricing)
- Wall-clock duration
- Output correctness (file existence + content)

In [ ]:
# --- Prompt definitions ---

SMALL_FILES = ["config_a.ini", "config_b.ini", "config_c.ini"]
MEDIUM_FILES = ["utils.py", "models.py", "views.py", "routes.py", "config.py"]
LARGE_FILES = [
    "utils.py", "models.py", "views.py", "routes.py", "config.py",
    "settings.ini", "test_utils.py", "test_models.py", "helpers.py", "constants.py",
]

EXPECTED_FILES = {
    "small": SMALL_FILES,
    "medium": MEDIUM_FILES,
    "large": LARGE_FILES,
}


def _direct_prompt(tier, out_dir):
    """Build a direct-write prompt for a given tier."""
    if tier == "small":
        return (
            f"Write exactly 3 config files to {out_dir}/. "
            f"Use the Write tool for each file. "
            f"File 1: {out_dir}/config_a.ini with content 'host=localhost'. "
            f"File 2: {out_dir}/config_b.ini with content 'port=8080'. "
            f"File 3: {out_dir}/config_c.ini with content 'debug=true'. "
            f"Do not explain, just write the files."
        )
    elif tier == "medium":
        return (
            f"Write exactly 5 Python files to {out_dir}/. "
            f"Use the Write tool for each file. Do not explain, just write. "
            f"File 1: {out_dir}/utils.py — a module with 2 functions: slugify(text) that lowercases and replaces spaces with hyphens, and truncate(text, max_len=100) that truncates with ellipsis. "
            f"File 2: {out_dir}/models.py — a module with a dataclass User(id: int, name: str, email: str) and a dataclass Post(id: int, title: str, author: User). "
            f"File 3: {out_dir}/views.py — a module with functions index() returning 'Welcome', user_detail(user_id: int) returning f'User {{user_id}}', and post_list() returning 'Posts'. "
            f"File 4: {out_dir}/routes.py — a module with a dict ROUTES mapping '/' to 'index', '/users' to 'user_detail', '/posts' to 'post_list'. "
            f"File 5: {out_dir}/config.py — a module with constants DATABASE_URL='sqlite:///db.sqlite3', SECRET_KEY='dev-secret', DEBUG=True."
        )
    else:  # large
        return (
            f"Write exactly 10 files to {out_dir}/. "
            f"Use the Write tool for each file. Do not explain, just write. "
            f"File 1: {out_dir}/utils.py — slugify(text) and truncate(text, max_len=100) functions. "
            f"File 2: {out_dir}/models.py — dataclasses User(id, name, email) and Post(id, title, author). "
            f"File 3: {out_dir}/views.py — functions index(), user_detail(user_id), post_list(). "
            f"File 4: {out_dir}/routes.py — ROUTES dict mapping paths to view names. "
            f"File 5: {out_dir}/config.py — DATABASE_URL, SECRET_KEY, DEBUG constants. "
            f"File 6: {out_dir}/settings.ini — INI file with [database] host=localhost, port=5432 and [app] debug=true. "
            f"File 7: {out_dir}/test_utils.py — 2 test functions: test_slugify and test_truncate using assert. "
            f"File 8: {out_dir}/test_models.py — 2 test functions: test_user_creation and test_post_creation using assert. "
            f"File 9: {out_dir}/helpers.py — format_date(dt) returning ISO format string, parse_bool(s) converting 'true'/'false' strings. "
            f"File 10: {out_dir}/constants.py — MAX_RETRIES=3, TIMEOUT_S=30, VERSION='1.0.0'."
        )


def _delegated_prompt(tier, out_dir):
    """Build a delegation prompt that instructs the parent to use Task with Haiku."""
    inner_prompt = _direct_prompt(tier, out_dir)
    return (
        f"Use the Task tool to delegate file writing to a subagent. "
        f"Set subagent_type to 'general-purpose', model to 'haiku', and the prompt to the following: "
        f"{inner_prompt} "
        f"Return whatever the task returns. Do not write any files yourself."
    )


# Preview the prompts
for tier in ["small", "medium", "large"]:
    out_dir = BASE_DIR / tier / "preview"
    print(f"\n{'='*60}")
    print(f"{tier.upper()} TIER — {len(EXPECTED_FILES[tier])} files")
    print(f"{'='*60}")
    print(f"Direct:    {_direct_prompt(tier, out_dir)[:120]}...")
    print(f"Delegated: {_delegated_prompt(tier, out_dir)[:120]}...")

In [ ]:
# --- Run experiments: 3 tiers × 2 patterns × N_RUNS ---

for tier in ["small", "medium", "large"]:
    expected = EXPECTED_FILES[tier]
    ALL_RESULTS[tier] = {"direct": [], "delegated": []}
    ALL_VERIFY[tier] = {"direct": [], "delegated": []}

    for pattern in ["direct", "delegated"]:
        print(f"\n{'='*60}")
        print(f"{tier.upper()} / {pattern.upper()} — {len(expected)} files × {N_RUNS} runs")
        print(f"{'='*60}")

        for run_i in range(N_RUNS):
            out_dir = BASE_DIR / tier / pattern / f"run{run_i+1}"
            clean_dir(out_dir)

            if pattern == "direct":
                prompt = _direct_prompt(tier, out_dir)
            else:
                prompt = _delegated_prompt(tier, out_dir)

            cr = run_claude(prompt, model="sonnet")
            usage = extract_usage(cr)
            usage["run"] = run_i + 1
            usage["returncode"] = cr.returncode
            usage["duration_s"] = cr.duration_s

            # Verify files
            vr = verify_files(out_dir, expected)
            found = sum(1 for v in vr.values() if v["exists"])
            valid = sum(1 for v in vr.values() if v["non_empty"])

            status = "OK" if cr.returncode == 0 else "FAIL"
            models_str = ", ".join(usage["models"])
            print(f"  Run {run_i+1}/{N_RUNS}: {status} — "
                  f"out={usage['outputTokens']} "
                  f"cache={usage['cacheReadInputTokens']} "
                  f"time={cr.duration_s:.1f}s "
                  f"files={found}/{len(expected)} valid={valid}/{len(expected)} "
                  f"models=[{models_str}]")

            ALL_RESULTS[tier][pattern].append(usage)
            ALL_VERIFY[tier][pattern].append(vr)

print("\n" + "="*60)
print("ALL EXPERIMENTS COMPLETE")
print("="*60)

In [ ]:
# --- Results: Token comparison ---

print("TOKEN COMPARISON — Direct vs Delegated")
print("=" * 80)

for tier in ["small", "medium", "large"]:
    direct = ALL_RESULTS[tier]["direct"]
    delegated = ALL_RESULTS[tier]["delegated"]

    print(f"\n--- {tier.upper()} ({len(EXPECTED_FILES[tier])} files) ---")
    print(f"  {'Metric':<12} {'Direct':>20} {'Delegated':>20} {'Delta':>12} {'Overhead%':>10}")
    print(f"  {'-'*12} {'-'*20} {'-'*20} {'-'*12} {'-'*10}")

    for metric, label in [
        ("inputTokens", "Input"),
        ("outputTokens", "Output"),
        ("cacheReadInputTokens", "Cache Read"),
        ("total", "Total"),
        ("num_turns", "Turns"),
    ]:
        d = summarize(direct, metric)
        t = summarize(delegated, metric)
        delta = t["mean"] - d["mean"]
        pct = (delta / d["mean"] * 100) if d["mean"] > 0 else float("inf")
        print(f"  {label:<12} {d['mean']:>8.0f} ± {d['stdev']:>7.0f}"
              f" {t['mean']:>8.0f} ± {t['stdev']:>7.0f}"
              f" {delta:>+12.0f} {pct:>+9.1f}%")

    # Per-model breakdown for delegated pattern
    print(f"\n  Delegated per-model breakdown (mean across runs):")
    model_agg = {}
    for r in delegated:
        for model, usage in r.get("model_usage", {}).items():
            if model not in model_agg:
                model_agg[model] = {"input": [], "output": [], "cache": []}
            model_agg[model]["input"].append(usage.get("inputTokens", 0))
            model_agg[model]["output"].append(usage.get("outputTokens", 0))
            model_agg[model]["cache"].append(usage.get("cacheReadInputTokens", 0))

    for model, agg in sorted(model_agg.items()):
        mi = statistics.mean(agg["input"])
        mo = statistics.mean(agg["output"])
        mc = statistics.mean(agg["cache"])
        print(f"    {model}: in={mi:.0f} out={mo:.0f} cache={mc:.0f}")

In [ ]:
# --- Results: Cost analysis ---

print("COST ANALYSIS — Per-model pricing applied")
print("=" * 80)

COST_RESULTS = {}

for tier in ["small", "medium", "large"]:
    direct = ALL_RESULTS[tier]["direct"]
    delegated = ALL_RESULTS[tier]["delegated"]

    # Average model_usage across runs, then price
    def avg_model_usage(results):
        """Average model_usage dicts across runs."""
        agg = {}
        for r in results:
            for model, usage in r.get("model_usage", {}).items():
                if model not in agg:
                    agg[model] = {"inputTokens": [], "outputTokens": [], "cacheReadInputTokens": []}
                agg[model]["inputTokens"].append(usage.get("inputTokens", 0))
                agg[model]["outputTokens"].append(usage.get("outputTokens", 0))
                agg[model]["cacheReadInputTokens"].append(usage.get("cacheReadInputTokens", 0))
        return {
            model: {
                "inputTokens": statistics.mean(vals["inputTokens"]),
                "outputTokens": statistics.mean(vals["outputTokens"]),
                "cacheReadInputTokens": statistics.mean(vals["cacheReadInputTokens"]),
            }
            for model, vals in agg.items()
        }

    d_avg = avg_model_usage(direct)
    t_avg = avg_model_usage(delegated)

    d_cost = multi_model_cost_estimate(d_avg)
    t_cost = multi_model_cost_estimate(t_avg)

    COST_RESULTS[tier] = {"direct": d_cost, "delegated": t_cost}

    print(f"\n--- {tier.upper()} ({len(EXPECTED_FILES[tier])} files) ---")
    print(f"  Direct cost: ${d_cost['grand_total']:.6f}")
    for model, mc in d_cost.items():
        if model == "grand_total":
            continue
        print(f"    {model}: in=${mc['input']:.6f} out=${mc['output']:.6f} cache=${mc['cache_read']:.6f} = ${mc['total']:.6f}")

    print(f"  Delegated cost: ${t_cost['grand_total']:.6f}")
    for model, mc in t_cost.items():
        if model == "grand_total":
            continue
        print(f"    {model}: in=${mc['input']:.6f} out=${mc['output']:.6f} cache=${mc['cache_read']:.6f} = ${mc['total']:.6f}")

    delta = t_cost["grand_total"] - d_cost["grand_total"]
    pct = (delta / d_cost["grand_total"] * 100) if d_cost["grand_total"] > 0 else float("inf")
    label = "SAVINGS" if delta < 0 else "OVERHEAD"
    print(f"  → {label}: ${abs(delta):.6f} ({pct:+.1f}%)")

# --- Summary table ---
print(f"\n{'='*80}")
print(f"COST SUMMARY")
print(f"{'='*80}")
print(f"  {'Tier':<8} {'Files':>5} {'Direct $':>12} {'Delegated $':>12} {'Delta $':>12} {'%':>8}")
print(f"  {'-'*8} {'-'*5} {'-'*12} {'-'*12} {'-'*12} {'-'*8}")

for tier in ["small", "medium", "large"]:
    d = COST_RESULTS[tier]["direct"]["grand_total"]
    t = COST_RESULTS[tier]["delegated"]["grand_total"]
    delta = t - d
    pct = (delta / d * 100) if d > 0 else float("inf")
    print(f"  {tier:<8} {len(EXPECTED_FILES[tier]):>5} ${d:>11.6f} ${t:>11.6f} ${delta:>+11.6f} {pct:>+7.1f}%")

# --- Opus projection ---
print(f"\n{'='*80}")
print("OPUS PROJECTION")
print("(Scale Sonnet findings by Opus/Sonnet price ratios)")
print(f"{'='*80}")
print(f"  Opus output is {25/15:.2f}× Sonnet output pricing")
print(f"  Opus cache is {0.50/0.30:.2f}× Sonnet cache pricing")
print(f"  Opus input is {5/3:.2f}× Sonnet input pricing")
print()

for tier in ["small", "medium", "large"]:
    d_avg = avg_model_usage(ALL_RESULTS[tier]["direct"])
    # For direct: re-price the Sonnet tokens at Opus rates
    opus_direct = 0.0
    for model, usage in d_avg.items():
        # Replace sonnet pricing with opus pricing
        opus_prices = PRICING["claude-opus-4-6"]
        opus_direct += usage["inputTokens"] / 1e6 * opus_prices["input"]
        opus_direct += usage["outputTokens"] / 1e6 * opus_prices["output"]
        opus_direct += usage["cacheReadInputTokens"] / 1e6 * opus_prices["cache_read"]

    # For delegated: parent tokens at Opus rates, Haiku tokens stay at Haiku rates
    t_avg = avg_model_usage(ALL_RESULTS[tier]["delegated"])
    opus_delegated = 0.0
    for model, usage in t_avg.items():
        if "haiku" in model.lower():
            prices = PRICING["claude-haiku-4-5-20251001"]
        else:
            prices = PRICING["claude-opus-4-6"]  # parent at Opus rates
        opus_delegated += usage["inputTokens"] / 1e6 * prices["input"]
        opus_delegated += usage["outputTokens"] / 1e6 * prices["output"]
        opus_delegated += usage["cacheReadInputTokens"] / 1e6 * prices["cache_read"]

    delta = opus_delegated - opus_direct
    pct = (delta / opus_direct * 100) if opus_direct > 0 else float("inf")
    label = "SAVES" if delta < 0 else "COSTS"
    print(f"  {tier}: Opus direct=${opus_direct:.6f}  delegated=${opus_delegated:.6f}  "
          f"{label} ${abs(delta):.6f} ({pct:+.1f}%)")

In [ ]:
# --- Results: Wall-clock timing ---

print("WALL-CLOCK TIMING")
print("=" * 80)
print(f"  {'Tier':<8} {'Files':>5} {'Direct (s)':>18} {'Delegated (s)':>18} {'Delta (s)':>12}")
print(f"  {'-'*8} {'-'*5} {'-'*18} {'-'*18} {'-'*12}")

for tier in ["small", "medium", "large"]:
    d = summarize(ALL_RESULTS[tier]["direct"], "duration_s")
    t = summarize(ALL_RESULTS[tier]["delegated"], "duration_s")
    delta = t["mean"] - d["mean"]
    print(f"  {tier:<8} {len(EXPECTED_FILES[tier]):>5}"
          f" {d['mean']:>7.1f} ± {d['stdev']:>5.1f}s"
          f" {t['mean']:>7.1f} ± {t['stdev']:>5.1f}s"
          f" {delta:>+11.1f}s")

print(f"\nNote: Delegation adds subprocess spawn overhead (parent → subagent),")
print(f"but Haiku generates tokens faster than Sonnet. Net effect depends on task size.")

In [ ]:
# --- Results: Correctness verification ---

print("CORRECTNESS VERIFICATION")
print("=" * 80)
print(f"  {'Tier':<8} {'Pattern':<11} {'Expected':>8} {'Found':>8} {'Valid':>8} {'Pass':>6}")
print(f"  {'-'*8} {'-'*11} {'-'*8} {'-'*8} {'-'*8} {'-'*6}")

for tier in ["small", "medium", "large"]:
    expected_count = len(EXPECTED_FILES[tier])
    for pattern in ["direct", "delegated"]:
        verifications = ALL_VERIFY[tier][pattern]
        # Average across runs
        found_counts = []
        valid_counts = []
        for vr in verifications:
            found_counts.append(sum(1 for v in vr.values() if v["exists"]))
            valid_counts.append(sum(1 for v in vr.values() if v["non_empty"]))

        avg_found = statistics.mean(found_counts)
        avg_valid = statistics.mean(valid_counts)
        all_pass = all(f == expected_count and v == expected_count
                       for f, v in zip(found_counts, valid_counts))

        print(f"  {tier:<8} {pattern:<11} {expected_count:>8}"
              f" {avg_found:>8.1f} {avg_valid:>8.1f}"
              f" {'YES' if all_pass else 'NO':>6}")

        # Show per-run detail if any failures
        if not all_pass:
            for run_i, vr in enumerate(verifications):
                missing = [f for f, v in vr.items() if not v["exists"]]
                empty = [f for f, v in vr.items() if v["exists"] and not v["non_empty"]]
                if missing or empty:
                    print(f"    Run {run_i+1}: missing={missing} empty={empty}")

## Conclusions

Tested on Claude Code, 2026-02-20. Sonnet 4.6 parent, 3 runs per experiment.

### Hypothesis verdicts

| # | Hypothesis | Verdict | Evidence |
|---|-----------|---------|----------|
| H1 | Delegation saves money for ≥5 files on Sonnet | **BARELY** — noise-level | 5 files: −0.5%; 10 files reverses to +9.4% |
| H2 | Break-even at ~2K output tokens (Sonnet), ~1.35K (Opus) | **WRONG** | Crossover at ~5 files (~600 output tokens), not ~2K |
| H3 | Wall-clock time is similar | **REFUTED** | +2s to +28s slower; medium tier had 86s outlier |
| H4 | Haiku produces correct file output | **UNTESTABLE** | Neither pattern wrote files (both 0/N) |

### Key findings

1. **Neither pattern successfully wrote files.** Output tokens for direct (~400–600) are too low
   to represent Write tool invocations. The agents responded conversationally — outputting the
   file content as text — rather than calling Write. The likely cause is that `--permission-mode default`
   does not auto-approve Write tools in non-interactive `claude -p` sessions. This invalidates H4
   and means the cost comparison reflects planning/describing work, not actual file creation.

2. **The cost crossover is at ~5 files, much earlier than predicted.** H2 estimated break-even at
   ~2,000 Sonnet output tokens; the actual crossover is at ~5 files (~600 total output tokens). The
   pre-experiment model overestimated how many output tokens direct writes would generate and
   underestimated how much Sonnet cache overhead dominates direct costs ($0.026 of $0.035 at medium
   tier is cache). The savings at the crossover are $0.000165 per invocation — noise-level.

3. **The break-even is non-monotonic.** Medium (5 files) reaches −0.5% savings; large (10 files)
   swings back to +9.4% overhead. As files increase, the Sonnet parent generates more result-
   processing output tokens (1032 at medium, 992 at large) because it processes a larger Task result,
   eroding the savings from cheaper Haiku output tokens.

4. **Delegation is slower, not similar.** Small: +1.9s. Medium: +28.2s (mean), with extreme
   variance — one run hit 86.1s. Large: +7.0s. The medium tier's high variance suggests occasional
   Haiku cold-start or scheduling delays. Delegation is never wall-clock neutral.

5. **Opus economics are more favorable.** At 5 files, delegation saves 5.8% on Opus vs 0.5% on
   Sonnet — roughly proportional to the wider output-price gap. But the absolute savings are still
   small ($0.003 per invocation) and the crossover remains narrow and non-monotonic.

### Break-even analysis

- **Actual crossover (Sonnet):** ~5 files / ~600 output tokens → −0.5% ($0.000165/invocation)
- **Pre-experiment prediction:** ~2,000 output tokens — **off by 3×**
- **Why the estimate was wrong:** Direct costs are dominated by cache reads (not output tokens),
  so the per-output-token savings of $10/MTok (Sonnet→Haiku) apply only to a small fraction of
  total cost. Meanwhile, delegation adds Sonnet parent output overhead that scales with task size.
- **Opus crossover:** Also at ~5 files (−5.8%), but same non-monotonicity at 10 files (+2.8%)

### Practical guidance

- **Do not use delegation to save money on file writes.** The savings at crossover are noise-level
  ($0.000165–$0.003 per invocation). The cost relationship is non-monotonic and fragile.
- **Delegation is slower.** Add 2–28s per invocation, with potential outliers. Avoid for latency-
  sensitive workflows.
- **File writing via `claude -p` requires `--permission-mode acceptEdits`.** Neither direct nor
  delegated patterns successfully wrote files with the default permission mode. Any future experiment
  measuring actual write behavior must set `permission_mode="acceptEdits"` in `run_claude()`.
- **The finding from `task-subagent-token-overhead.ipynb` holds here too:** direct `claude -p`
  is more powerful and cheaper than Task delegation. For write-heavy work, use direct invocation
  with `acceptEdits` mode — don't delegate to Haiku.

In [ ]:
# --- Raw data dump ---

raw_data = {
    "metadata": {
        "date": "2026-02-20",
        "parent_model": "sonnet",
        "delegation_model": "haiku",
        "n_runs": N_RUNS,
        "tiers": {tier: EXPECTED_FILES[tier] for tier in EXPECTED_FILES},
    },
    "results": {},
    "verification": {},
    "cost_analysis": {},
}

for tier in ["small", "medium", "large"]:
    raw_data["results"][tier] = ALL_RESULTS[tier]
    raw_data["verification"][tier] = ALL_VERIFY[tier]
    if tier in COST_RESULTS:
        raw_data["cost_analysis"][tier] = {
            "direct": COST_RESULTS[tier]["direct"],
            "delegated": COST_RESULTS[tier]["delegated"],
        }

print(json.dumps(raw_data, indent=2, default=str))